# RobotAI4B — Colab'da LoRA ince ayarı

Bu defter Gemma 4 E4B modelini Türkçe robot asistan verisiyle LoRA ile eğitir ve Raspberry Pi 5 için GGUF dosyası üretir.

**Çalışma zamanı:** `Çalışma zamanı > Çalışma zamanı türünü değiştir`
- **A100 / L4 (önerilen):** bf16 LoRA, en iyi kalite.
- **T4 (ücretsiz):** `LOAD_IN_4BIT = True` (QLoRA). Ücretsiz Colab'da RAM ~12 GB olduğu için birleştirme yerine **Seçenek B** (sadece LoRA GGUF) kullanın.

Depo özel (private) ise klonlama satırında GitHub kişisel erişim jetonu (token) gerekir.

In [ ]:
!git clone https://github.com/garippadam/RobotAI4B
%cd RobotAI4B
!pip install -q -r requirements-train.txt

In [ ]:
import torch
print(torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())
LOAD_IN_4BIT = not torch.cuda.is_bf16_supported()   # T4 -> True

## 1) Veriyi hazırla
Kendi örneklerinizi `data/robotai_persona.jsonl` dosyasına ekleyebilirsiniz.

In [ ]:
!python scripts/prepare_data.py --config configs/robotai4b.yaml

## 2) LoRA eğitimi

In [ ]:
flag = "--load-in-4bit" if LOAD_IN_4BIT else ""
!python scripts/train_lora.py --config configs/robotai4b.yaml {flag}

## 3a) Seçenek A — Birleştir + GGUF (≥ 25 GB RAM gerekir, ör. A100 yüksek RAM)

In [ ]:
!python scripts/merge_lora.py --adapter outputs/robotai4b-lora/final --out outputs/robotai4b-merged
!bash scripts/export_gguf.sh outputs/robotai4b-merged outputs/gguf Q4_K_M Q4_0

## 3b) Seçenek B — Sadece LoRA adaptörünü GGUF'a çevir (az RAM yeterli)

In [ ]:
!bash scripts/export_lora_gguf.sh outputs/robotai4b-lora/final outputs/gguf

## 4) Hugging Face'e yükle (Pi oradan indirecek)
Önce https://huggingface.co/settings/tokens adresinden *write* yetkili bir jeton alın.

In [ ]:
from huggingface_hub import login, HfApi
login()
REPO = "KULLANICI_ADINIZ/robotai4b-GGUF"   # <- değiştirin
api = HfApi()
api.create_repo(REPO, exist_ok=True, private=True)
import glob
for f in glob.glob("outputs/gguf/*.gguf"):
    if "BF16" in f:   # büyük ara dosyayı yükleme
        continue
    print("yükleniyor:", f)
    api.upload_file(path_or_fileobj=f, path_in_repo=f.split("/")[-1], repo_id=REPO)